In [1]:
import pandas as pd

In [5]:
df = pd.read_csv("../data/processed/rideshare_feature_engineering.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 637976 entries, 0 to 637975
Data columns (total 49 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   hour                     637976 non-null  int64  
 1   day                      637976 non-null  int64  
 2   month                    637976 non-null  int64  
 3   source                   637976 non-null  object 
 4   destination              637976 non-null  object 
 5   cab_type                 637976 non-null  object 
 6   name                     637976 non-null  object 
 7   price                    637976 non-null  float64
 8   distance                 637976 non-null  float64
 9   surge_multiplier         637976 non-null  float64
 10  latitude                 637976 non-null  float64
 11  longitude                637976 non-null  float64
 12  temperature              637976 non-null  float64
 13  apparentTemperature      637976 non-null  float64
 14  shor

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [8]:
features = [
    'tier_enc', 'name_enc', 'distance', 'surge_multiplier', 
    'surge_x_distance', 'is_surge', 'source_enc', 'destination_enc',
    'cab_type_enc', 'route_frequency', 'temperature', 'humidity'
]

X = df[features]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)


y_pred = rf_model.predict(X_test)

print(f"RMSE: {mean_squared_error(y_test, y_pred):.2f} USD")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f} USD")

RMSE: 3.24 USD
MAE: 1.14 USD


In [14]:
from sklearn.model_selection import RandomizedSearchCV

In [15]:
# ปรับ n_jobs เป็น 1 หรือ 2 และลด n_iter/cv ลงเล็กน้อย
rf_random = RandomizedSearchCV(
    estimator=rf_model, 
    param_distributions=param_dist, 
    n_iter=5,          # ลดจำนวนรอบการสุ่มลง
    cv=2,              # ลดจำนวน Fold ในการตรวจผล
    random_state=42, 
    n_jobs=1,          # ใช้แค่ 1 Core เพื่อประหยัด RAM
    verbose=2          # เปิดดูสถานะการทำงาน
)
rf_random.fit(X_train, y_train)

print(f"Best Params: {rf_random.best_params_}")

Fitting 2 folds for each of 5 candidates, totalling 10 fits
[CV] END max_depth=30, max_features=None, min_samples_split=5, n_estimators=300; total time=  12.9s
[CV] END max_depth=30, max_features=None, min_samples_split=5, n_estimators=300; total time=  12.5s
[CV] END max_depth=10, max_features=log2, min_samples_split=2, n_estimators=200; total time=   1.8s
[CV] END max_depth=10, max_features=log2, min_samples_split=2, n_estimators=200; total time=   1.8s
[CV] END max_depth=10, max_features=sqrt, min_samples_split=5, n_estimators=200; total time=   1.7s
[CV] END max_depth=10, max_features=sqrt, min_samples_split=5, n_estimators=200; total time=   1.7s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=300; total time=   6.9s
[CV] END max_depth=None, max_features=sqrt, min_samples_split=2, n_estimators=300; total time=   6.9s
[CV] END max_depth=30, max_features=sqrt, min_samples_split=10, n_estimators=300; total time=   4.8s
[CV] END max_depth=30, max_features

In [ ]:
best_rf = RandomForestRegressor(
    n_estimators=300, 
    min_samples_split=10, 
    max_features='sqrt', 
    max_depth=30, 
    random_state=42, 
    n_jobs=-1
)

best_rf.fit(X_train, y_train)

y_pred_best = best_rf.predict(X_test)

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

print("--- Improved Random Forest Results ---")
print(f"MAE: {mean_absolute_error(y_test, y_pred_best):.2f} USD")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_best)):.2f} USD")

--- Improved Random Forest Results ---
MAE: 1.08 USD
RMSE: 1.68 USD


In [17]:
import xgboost as xgb

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42
)

xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
print("--- XGBoost Results ---")
print(f"MAE: {mean_absolute_error(y_test, y_pred_xgb):.2f} USD")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_xgb)):.2f} USD")


--- XGBoost Results ---
MAE: 1.04 USD
RMSE: 1.59 USD


In [ ]:
y_train_log = np.log1p(y_train)

xgb_model.fit(X_train, y_train_log)

y_pred_log = xgb_model.predict(X_test)
y_pred_log = np.expm1(y_pred_log)

print(f"MAE: {mean_absolute_error(y_test, y_pred_log):.2f} USD")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_log)):.2f} USD")

MAE: 1.03 USD
RMSE: 1.60 USD


In [21]:
import lightgbm as lgb

In [ ]:
lgbm_model = lgb.LGBMRegressor(
    n_estimators=2000, 
    learning_rate=0.03, 
    num_leaves=64,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    n_jobs=-1,
    random_state=42
)

lgbm_model.fit(X_train, y_train)

y_pred_lgb = lgbm_model.predict(X_test)
print(f"LightGBM RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lgb)):.2f} USD")

[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005483 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 908
[LightGBM] [Info] Number of data points in the train set: 510380, number of used features: 12
[LightGBM] [Info] Start training from score 16.545838
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_

In [23]:
from sklearn.ensemble import VotingRegressor

In [ ]:
voted_model = VotingRegressor([
    ('rf', best_rf),
    ('xgb', xgb_model),
    ('lgbm', lgbm_model)
])

voted_model.fit(X_train, y_train)
y_pred_voted = voted_model.predict(X_test)

print(f"Voting RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_voted)):.2f} USD")

[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003815 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 908
[LightGBM] [Info] Number of data points in the train set: 510380, number of used features: 12
[LightGBM] [Info] Start training from score 16.545838
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_

In [26]:
from catboost import CatBoostRegressor

In [ ]:
cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function='RMSE',
    random_seed=42,
    verbose=100
)

cat_model.fit(X_train, y_train)
y_pred_cat = cat_model.predict(X_test)

print(f"MAE: {mean_absolute_error(y_test, y_pred_cat):.2f} USD")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_cat)):.2f} USD")

0:	learn: 8.9156221	total: 198ms	remaining: 3m 17s
100:	learn: 1.8319054	total: 1.62s	remaining: 14.4s
200:	learn: 1.7296819	total: 2.98s	remaining: 11.9s
300:	learn: 1.6958941	total: 4.29s	remaining: 9.96s
400:	learn: 1.6768696	total: 5.58s	remaining: 8.34s
500:	learn: 1.6631098	total: 6.87s	remaining: 6.84s
600:	learn: 1.6532588	total: 8.16s	remaining: 5.42s
700:	learn: 1.6452533	total: 9.46s	remaining: 4.04s
800:	learn: 1.6384000	total: 10.7s	remaining: 2.66s
900:	learn: 1.6321282	total: 12s	remaining: 1.31s
999:	learn: 1.6271795	total: 13.2s	remaining: 0us
MAE: 1.07 USD
RMSE: 1.63 USD


In [ ]:
cat_model.fit(X_train, y_train_log)

y_pred_log = cat_model.predict(X_test)

y_pred_final = np.expm1(y_pred_log)

print(f"MAE (with Log): {mean_absolute_error(y_test, y_pred_final):.2f} USD")
print(f"RMSE (with Log): {np.sqrt(mean_squared_error(y_test, y_pred_final)):.2f} USD")

0:	learn: 0.5032333	total: 34.1ms	remaining: 34.1s
100:	learn: 0.1169825	total: 1.38s	remaining: 12.3s
200:	learn: 0.1124560	total: 2.69s	remaining: 10.7s
300:	learn: 0.1109981	total: 4s	remaining: 9.3s
400:	learn: 0.1100686	total: 5.32s	remaining: 7.95s
500:	learn: 0.1093803	total: 6.63s	remaining: 6.6s
600:	learn: 0.1089056	total: 7.93s	remaining: 5.27s
700:	learn: 0.1085293	total: 9.24s	remaining: 3.94s
800:	learn: 0.1082050	total: 10.6s	remaining: 2.62s
900:	learn: 0.1079080	total: 11.8s	remaining: 1.3s
999:	learn: 0.1076482	total: 13.1s	remaining: 0us
MAE (with Log): 1.06 USD
RMSE (with Log): 1.64 USD


In [ ]:
df['is_rush_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
df['is_weekend']   = df['day'].isin([5, 6]).astype(int)
df['is_night']     = (df['hour'] >= 22) | (df['hour'] <= 5)
df['is_night']     = df['is_night'].astype(int)

df['source_price_enc'] = df['source'].map(df.groupby('source')['price'].mean())
df['dest_price_enc']   = df['destination'].map(df.groupby('destination')['price'].mean())

df['precip_x_wind'] = df['precipProbability'] * df['windSpeed']  # interaction

features = [
    'tier_enc', 'name_enc', 'distance', 'surge_multiplier',
    'surge_x_distance', 'is_surge', 'cab_type_enc', 'route_frequency',
    'temperature', 'humidity',

    'hour', 'is_rush_hour', 'is_weekend', 'is_night',

    'source_price_enc', 'dest_price_enc',

    'precipProbability', 'windSpeed', 'uvIndex', 'cloudCover',
    'precip_x_wind',
]

X = df[features]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [32]:
cat_model.fit(X_train, y_train_log)

y_pred_cat = np.expm1(cat_model.predict(X_test))

print(f"MAE:  {mean_absolute_error(y_test, y_pred_cat):.2f} USD")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_cat)):.2f} USD")

0:	learn: 0.5030609	total: 26.9ms	remaining: 26.9s
100:	learn: 0.1170138	total: 1.48s	remaining: 13.2s
200:	learn: 0.1123053	total: 2.87s	remaining: 11.4s
300:	learn: 0.1107724	total: 4.25s	remaining: 9.87s
400:	learn: 0.1098927	total: 5.63s	remaining: 8.41s
500:	learn: 0.1092805	total: 7s	remaining: 6.97s
600:	learn: 0.1087840	total: 8.37s	remaining: 5.56s
700:	learn: 0.1083768	total: 9.72s	remaining: 4.15s
800:	learn: 0.1080361	total: 11s	remaining: 2.74s
900:	learn: 0.1077185	total: 12.4s	remaining: 1.36s
999:	learn: 0.1074354	total: 13.7s	remaining: 0us
MAE:  1.07 USD
RMSE: 1.64 USD


In [33]:
importance = pd.Series(
    cat_model.get_feature_importance(),
    index=features
).sort_values(ascending=False)

print(importance)

name_enc             40.479040
tier_enc             31.313311
surge_x_distance     12.918323
cab_type_enc         11.491736
distance              1.567753
source_price_enc      0.515486
route_frequency       0.455098
dest_price_enc        0.451180
surge_multiplier      0.414283
is_surge              0.279892
windSpeed             0.030055
temperature           0.021382
humidity              0.017309
cloudCover            0.016190
hour                  0.014405
precip_x_wind         0.004430
is_rush_hour          0.003657
precipProbability     0.003274
uvIndex               0.002062
is_night              0.001135
is_weekend            0.000000
dtype: float64


In [ ]:
features_sub = [
    'distance', 'surge_multiplier', 'surge_x_distance', 'is_surge',
    'source_price_enc', 'dest_price_enc', 'route_frequency',
    'hour', 'is_rush_hour', 'temperature', 'humidity'
]

results = {}

for tier in ['Economy', 'Mid', 'Premium']:
    mask_train = X_train.index.isin(df[df['tier'] == tier].index)
    mask_test  = X_test.index.isin(df[df['tier'] == tier].index)

    X_tr = X_train.loc[mask_train, features_sub]
    X_te = X_test.loc[mask_test,  features_sub]
    y_tr = y_train[mask_train]
    y_te = y_test[mask_test]

    model = CatBoostRegressor(
        iterations=1000, learning_rate=0.05,
        depth=8, random_seed=42, verbose=0
    )
    model.fit(X_tr, np.log1p(y_tr))
    pred = np.expm1(model.predict(X_te))

    mae  = mean_absolute_error(y_te, pred)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    results[tier] = {'MAE': mae, 'RMSE': rmse}
    print(f"{tier:10s} → MAE: {mae:.2f}  RMSE: {rmse:.2f}")

Economy    → MAE: 1.10  RMSE: 1.61
Mid        → MAE: 2.82  RMSE: 3.45
Premium    → MAE: 5.20  RMSE: 6.01


ValueError: need at least one array to concatenate

In [35]:
print(df.groupby('tier')['price'].agg(['count', 'mean', 'std', 'min', 'max']))

          count       mean       std   min   max
tier                                            
Economy  106324   7.440592  2.507983   2.5  42.5
Mid      267756  12.013185  4.464858   5.0  76.0
Premium  263896  24.811581  8.053402  10.5  97.5


In [37]:
import optuna

c:\Users\Songwit Rueangsawat\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
features = [
    'tier_enc', 'name_enc', 'distance', 'surge_multiplier',
    'surge_x_distance', 'is_surge', 'cab_type_enc', 'route_frequency',
    'source_price_enc', 'dest_price_enc',
    'temperature', 'humidity', 'hour', 'is_rush_hour'
]

X = df[features]
y = np.log1p(df['price'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def objective(trial):
    params = {
        'iterations':    trial.suggest_int('iterations', 500, 3000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth':         trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg':   trial.suggest_float('l2_leaf_reg', 1, 10),
        'random_seed': 42,
        'verbose': 0
    }
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return np.sqrt(mean_squared_error(y_test, pred))

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)

print("Best params:", study.best_params)
print("Best RMSE (log scale):", study.best_value)

[I 2026-04-18 05:43:11,138] A new study created in memory with name: no-name-18bf703b-a1b6-4d83-bdd6-7ccc82ddf926
[I 2026-04-18 05:43:39,276] Trial 0 finished with value: 0.10742314931188503 and parameters: {'iterations': 2780, 'learning_rate': 0.11996767549893808, 'depth': 5, 'l2_leaf_reg': 9.274182124739463}. Best is trial 0 with value: 0.10742314931188503.
[I 2026-04-18 05:43:50,076] Trial 1 finished with value: 0.10999291008113428 and parameters: {'iterations': 1164, 'learning_rate': 0.06666797617562123, 'depth': 4, 'l2_leaf_reg': 1.9861531660878806}. Best is trial 0 with value: 0.10742314931188503.
[I 2026-04-18 05:44:31,359] Trial 2 finished with value: 0.10689420963157394 and parameters: {'iterations': 1617, 'learning_rate': 0.04759297673498495, 'depth': 10, 'l2_leaf_reg': 4.521766388141034}. Best is trial 2 with value: 0.10689420963157394.
[I 2026-04-18 05:45:02,868] Trial 3 finished with value: 0.10673149679816628 and parameters: {'iterations': 1955, 'learning_rate': 0.2875916

Best params: {'iterations': 2559, 'learning_rate': 0.08733609960108993, 'depth': 9, 'l2_leaf_reg': 3.313582787591232}
Best RMSE (log scale): 0.10644841712242131


In [39]:
best = study.best_params
final_model = CatBoostRegressor(**best, random_seed=42, verbose=100)
final_model.fit(X_train, y_train)

y_pred_final = np.expm1(final_model.predict(X_test))
y_test_actual = np.expm1(y_test)

print(f"MAE:  {mean_absolute_error(y_test_actual, y_pred_final):.2f} USD")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_actual, y_pred_final)):.2f} USD")

0:	learn: 0.4854397	total: 30ms	remaining: 1m 16s
100:	learn: 0.1120309	total: 1.68s	remaining: 40.8s
200:	learn: 0.1096560	total: 3.28s	remaining: 38.5s
300:	learn: 0.1084971	total: 4.91s	remaining: 36.9s
400:	learn: 0.1076914	total: 6.49s	remaining: 34.9s
500:	learn: 0.1070231	total: 8.09s	remaining: 33.2s
600:	learn: 0.1064391	total: 9.66s	remaining: 31.5s
700:	learn: 0.1058790	total: 11.2s	remaining: 29.7s
800:	learn: 0.1053566	total: 12.8s	remaining: 28s
900:	learn: 0.1048972	total: 14.3s	remaining: 26.3s
1000:	learn: 0.1044306	total: 15.8s	remaining: 24.6s
1100:	learn: 0.1040276	total: 17.4s	remaining: 23s
1200:	learn: 0.1036121	total: 18.9s	remaining: 21.4s
1300:	learn: 0.1032405	total: 20.4s	remaining: 19.8s
1400:	learn: 0.1028837	total: 22s	remaining: 18.2s
1500:	learn: 0.1025378	total: 23.5s	remaining: 16.6s
1600:	learn: 0.1022148	total: 25.2s	remaining: 15.1s
1700:	learn: 0.1018701	total: 26.9s	remaining: 13.6s
1800:	learn: 0.1015687	total: 28.6s	remaining: 12s
1900:	learn: 

In [ ]:
import joblib
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    final_model, X_train, y_train,
    cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1
)

cv_rmse = -cv_scores
print(f"CV RMSE (log scale): {cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}")

results_df = X_test.copy()
results_df['actual']    = y_test_actual.values
results_df['predicted'] = y_pred_final
results_df['tier']      = df.loc[X_test.index, 'tier'].values

for tier in ['Economy', 'Mid', 'Premium']:
    subset = results_df[results_df['tier'] == tier]
    mae  = mean_absolute_error(subset['actual'], subset['predicted'])
    rmse = np.sqrt(mean_squared_error(subset['actual'], subset['predicted']))
    print(f"{tier:10s} → MAE: {mae:.2f}  RMSE: {rmse:.2f}  (n={len(subset):,})")

joblib.dump(final_model, 'catboost_rideshare_final.pkl')
joblib.dump(features, 'features_list.pkl')
print("\nบันทึก model เรียบร้อยแล้ว ✅")

CV RMSE (log scale): 0.1076 ± 0.0004
Economy    → MAE: 1.02  RMSE: 1.47  (n=21,254)
Mid        → MAE: 0.94  RMSE: 1.62  (n=53,615)
Premium    → MAE: 1.15  RMSE: 1.63  (n=52,727)

บันทึก model เรียบร้อยแล้ว ✅
